# Input Data and Linear Regression in TensorFlow

## Introduction

In this notebook, we will explore how to handle input data in TensorFlow and build a fundamental machine learning model: **Linear Regression**. 

We will cover:
1.  **Importing Data:** How to load data using Pandas and convert it for TensorFlow.
2.  **Loss Functions:** Understanding MSE, MAE, and Huber loss, and why we need them.
3.  **Linear Regression:** Building, training, and optimizing a linear model.
4.  **Batch Training:** Handling large datasets efficiently.

## Part 1: Importing Data

Data often comes in CSV files. While TensorFlow has its own data loading pipelines (`tf.data`), a common workflow for smaller to medium datasets is to load them with **Pandas** and then convert them to **NumPy** arrays or **TensorFlow** tensors.

In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
filepath = '../../00-data/KC House Data.csv'
housing = pd.read_csv(filepath)

# Inspect the data
print(housing.head())

ModuleNotFoundError: No module named 'tensorflow'

### Data Formatting
TensorFlow expects data to be in numerical format (usually `float32`). We often need to cast columns to the correct type.

In [ ]:
# Selecting features and target
# Let's predict 'price' based on 'sqft_living'

# Convert to numpy array and cast to float32
price = np.array(housing['price'], dtype=np.float32)
sqft_living = np.array(housing['sqft_living'], dtype=np.float32)

# Alternatively, convert pandas Series to tensor properly
# Note: Use .values to extract numpy array first, then convert to tensor
price_tf = tf.constant(housing['price'].values, dtype=tf.float32)
sqft_living_tf = tf.constant(housing['sqft_living'].values, dtype=tf.float32)

print(f"Price shape: {price.shape}")
print(f"Sqft Living shape: {sqft_living.shape}")

### Data Normalization - CRITICAL FOR TRAINING

**Why normalize?**
- `price` ranges from ~100K to millions (mean ~540K)
- `sqft_living` ranges from ~300 to ~13K (mean ~2,080)
- These different scales cause **massive gradient problems**

**Problem without normalization:**
```
loss = (500,000 - prediction)² = 250,000,000,000  # Billions!
gradient ≈ -2 * error * feature ≈ -2,000,000,000  # Huge gradients!
```

**Solution: Z-score normalization**
- Transform to mean=0, std=1
- Gradients become stable
- Faster convergence
- All features contribute equally

**Formula:** `normalized = (original - mean) / std`

In [ ]:
# Normalize the data for better training
price_mean = np.mean(price)
price_std = np.std(price)
sqft_mean = np.mean(sqft_living)
sqft_std = np.std(sqft_living)

# Standardize (z-score normalization)
price_norm = (price - price_mean) / price_std
sqft_living_norm = (sqft_living - sqft_mean) / sqft_std

print(f"Original Price - Mean: {price_mean:.2f}, Std: {price_std:.2f}")
print(f"Original Sqft Living - Mean: {sqft_mean:.2f}, Std: {sqft_std:.2f}")
print(f"\nNormalized Price - Mean: {np.mean(price_norm):.6f}, Std: {np.std(price_norm):.6f}")
print(f"Normalized Sqft Living - Mean: {np.mean(sqft_living_norm):.6f}, Std: {np.std(sqft_living_norm):.6f}")

In [ ]:
# Example calculation
y_true = tf.constant([10.0, 20.0, 30.0])
y_pred = tf.constant([12.0, 18.0, 35.0])

mse = tf.keras.losses.mse(y_true, y_pred)
mae = tf.keras.losses.mae(y_true, y_pred)
huber = tf.keras.losses.Huber()(y_true, y_pred)

print(f"MSE: {mse.numpy()}")
print(f"MAE: {mae.numpy()}")
print(f"Huber: {huber.numpy()}")

In [ ]:
# Define trainable variables initialized with a guess
intercept = tf.Variable(0.1, dtype=tf.float32)
slope = tf.Variable(0.1, dtype=tf.float32)

# Define the model (Forward pass)
def linear_regression(intercept, slope, features):
    return intercept + slope * features

In [ ]:
def compute_loss(intercept, slope, targets, features):
    predictions = linear_regression(intercept, slope, features)
    return tf.keras.losses.mse(targets, predictions)

# Optimizer
opt = tf.keras.optimizers.Adam(learning_rate=0.1)

In [ ]:
losses = []

print("Training Started...")
for i in range(1000):
    # Use GradientTape to compute gradients
    with tf.GradientTape() as tape:
        current_loss = compute_loss(intercept, slope, price, sqft_living)

    # Compute gradients with respect to intercept and slope
    gradients = tape.gradient(current_loss, [intercept, slope])

    # Apply gradients to update variables
    opt.apply_gradients(zip(gradients, [intercept, slope]))

    # Track loss for visualization
    losses.append(float(current_loss.numpy()))

    if i % 100 == 0:
        print(f"Epoch {i}: Loss: {current_loss.numpy():.2f}")

print(f"\nFinal Parameters: Intercept={intercept.numpy():.2f}, Slope={slope.numpy():.2f}")

In [ ]:
plt.figure(figsize=(10, 6))

# Plot original data (not normalized)
plt.scatter(sqft_living, price, alpha=0.3, label='Data')

# Make predictions on normalized data, then DENORMALIZE for plotting
predictions_norm = linear_regression(intercept, slope, sqft_living_norm)
# Reverse normalization: original = normalized * std + mean
predictions_original = predictions_norm * price_std + price_mean

plt.plot(sqft_living, predictions_original, color='red', linewidth=3, label='Regression Line')
plt.xlabel('Sqft Living')
plt.ylabel('Price ($)')
plt.title('Linear Regression: Price vs Sqft Living')
plt.legend()
plt.show()

# Plot Loss Curve
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title('Loss over Epochs (on normalized scale)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nFinal loss: {losses[-1]:.4f}")
print("Note: Loss is calculated on normalized data (mean=0, std=1)")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(sqft_living, price, alpha=0.3, label='Data')
plt.plot(sqft_living, linear_regression(intercept, slope, sqft_living), color='red', linewidth=3, label='Regression Line')
plt.xlabel('Sqft Living')
plt.ylabel('Price')
plt.title('Linear Regression: Price vs Sqft Living')
plt.legend()
plt.show()

# Plot Loss Curve
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.show()

In [ ]:
# Re-initialize variables for a clean start
intercept_batch = tf.Variable(0.0, dtype=tf.float32)
slope_batch = tf.Variable(0.0, dtype=tf.float32)
opt_batch = tf.keras.optimizers.Adam(learning_rate=0.01)

print("Batch Training with Normalization")
print("=" * 50)

# Batch training loop
for epoch in range(5):  # Fewer epochs for demo
    for batch in pd.read_csv(filepath, chunksize=1000):
        # Extract batch features and targets
        price_batch = np.array(batch['price'], dtype=np.float32)
        size_batch = np.array(batch['sqft_living'], dtype=np.float32)
        
        # CRITICAL: Normalize batch data using the SAME mean/std from training set
        # Never compute new statistics per batch - this would break the model!
        price_batch_norm = (price_batch - price_mean) / price_std
        size_batch_norm = (size_batch - sqft_mean) / sqft_std
        
        # Train on this batch using GradientTape
        with tf.GradientTape() as tape:
            loss = compute_loss(intercept_batch, slope_batch, price_batch_norm, size_batch_norm)
        
        # Compute and apply gradients
        gradients = tape.gradient(loss, [intercept_batch, slope_batch])
        opt_batch.apply_gradients(zip(gradients, [intercept_batch, slope_batch]))
    
    print(f"Epoch {epoch} complete.")

print(f"\nBatch Training Params (normalized scale):")
print(f"Intercept={intercept_batch.numpy():.4f}, Slope={slope_batch.numpy():.4f}")

## Exercises

1. **Load a different feature:** Modify the code to predict `price` using `bedrooms` instead of `sqft_living`.
   - Don't forget to normalize the `bedrooms` feature!
   
2. **Experiment with Loss:** Try training the model using `tf.keras.losses.mean_absolute_error` instead of MSE. 
   - How does the result change?
   - Which converges faster?
   
3. **Visualizing Batch Training:** Modify the batch training loop to record the loss after every batch, and plot the loss curve.
   - Hint: Create a list and append loss values inside the inner loop

4. **Try without normalization:** Comment out the normalization and train on raw `price` and `sqft_living`.
   - What happens to the loss values?
   - Does the model converge?

## 2. Exploratory Data Analysis (EDA)

Before preprocessing, it's crucial to understand the data structure, types, and summary statistics.

## Key Takeaways

### 1. Data Normalization is CRITICAL
- **Why?** Features with different scales cause unstable gradients
- **Solution:** Z-score normalization (mean=0, std=1)
- **Impact:** Loss went from billions to < 1.0, model actually learns!

### 2. TensorFlow 2.x API Updates
- Use `tf.keras.losses.mean_squared_error` (not `mse`)
- Use `GradientTape` for custom training loops (not `optimizer.minimize()`)
- Always `tf.reduce_mean()` on loss for scalar gradients

### 3. Proper Training Workflow
```python
1. Load data → 2. Normalize → 3. Train → 4. Denormalize predictions
```

### 4. Learning Rate Matters
- With normalized data: 0.01 works well
- Too high (0.1): May overshoot and diverge
- Too low (0.001): Slow convergence

### 5. Batch Training Best Practice
- **Always use the SAME normalization statistics** (mean/std from training set)
- Never compute new statistics per batch
- This ensures consistency across all data

---

**Compare:**
- **Without normalization:** Loss = 426,000,000,000 → Unusable
- **With normalization:** Loss = ~0.3-0.5 → Model learns properly!

In [ ]:
# Your Exercise Solution Here